# Benchmark Aggregation — Validation Matrices
Generate `_modified_features.csv` files from validation matrix benchmark results.

In [1]:
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'

header_names = ['matrix_name','distribution','placement','seed',
                'nr_rows','nr_cols','nr_nzeros','density','mem_footprint','mem_range',
                'avg_nnz_per_row','std_nnz_per_row',
                'avg_bw','std_bw','avg_bw_scaled','std_bw_scaled',
                'avg_sc','std_sc','avg_sc_scaled','std_sc_scaled',
                'skew','avg_num_neighbours','cross_row_similarity',
                'format_name','time','gflops','W_avg','J_estimated', 'System', 'Arch']

# Extended header for final merge (includes bandwidth columns)
header_names_ext = header_names + ['mem_bw_gbytes_s', 'cache_bw_gbytes_s']

def find_class(mem_footprint):
    low_mb_list =  [4,8,16,32,64,128,256,512,1024,2048,4096]
    high_mb_list = [8,16,32,64,128,256,512,1024,2048,4096,8192]
    for i in range(len(low_mb_list)):
        if mem_footprint >= low_mb_list[i] and mem_footprint <= high_mb_list[i]:
            return '[' + str(low_mb_list[i]) + '-' + str(high_mb_list[i]) + ']'
    return str(-1)

# Thread counts per CPU system
Hawk_threads     = 64
Epyc_threads     = 24
Xeon_threads     = 14
Icy_threads      = 16
Sapphire_threads = 56
Arm_threads      = 80
Grace_threads    = 72
Power9_threads   = 32

In [2]:
# Column schemas for different CSV layouts
SCHEMA_OLD = ['matrix_name','distribution','placement','seed',
              'nr_rows','nr_cols','nr_nzeros','density','mem_footprint','mem_range',
              'avg_nnz_per_row','std_nnz_per_row',
              'avg_bw','std_bw','avg_sc','std_sc',
              '1','2','3','4','5','6','7',
              'format_name','time','gflops','W_avg','J_estimated']

SCHEMA_V100 = ['matrix_name','distribution','placement','diagonal_factor','seed',
               'nr_rows','nr_cols','nr_nzeros','density','mem_footprint','mem_range',
               'avg_nnz_per_row','std_nnz_per_row',
               'avg_bw','std_bw','avg_sc','std_sc',
               'format_name','time','gflops','W_avg','J_estimated']

SCHEMA_NEW = ['matrix_name','distribution','placement','seed',
              'nr_rows','nr_cols','nr_nzeros','density','mem_footprint','mem_range',
              'avg_nnz_per_row','std_nnz_per_row',
              'avg_bw','std_bw','avg_bw_scaled','std_bw_scaled',
              'avg_sc','std_sc','avg_sc_scaled','std_sc_scaled',
              'skew','avg_num_neighbours','cross_row_similarity',
              'format_name','time','gflops','W_avg','J_estimated','System']

In [3]:
def create_complete_csv(dataframe, system, arch, fname, seed_from_data=True):
    """Unified function for GPU/CPU validation matrix CSV creation.

    Replaces the old create_complete_gpu_csv and create_complete_cpu_csv.
    Uses list-of-dicts to avoid FutureWarning from pd.concat with empty DataFrames.
    """
    dataframe['matrix_name'] = dataframe['matrix_name'].apply(
        lambda x: x.split('/')[-1].split('.')[0]
    )
    vm_features = pd.read_csv('../benchmark_results/validation_matrices_features.csv', sep='\t')
    features_by_matrix = vm_features.set_index('matrix').to_dict('index')
    matrix_names = list(vm_features['matrix'])

    rows = []
    for matrix_name in matrix_names:
        feat = features_by_matrix[matrix_name]
        for _, curr in dataframe[dataframe['matrix_name'] == matrix_name].iterrows():
            m, n, nz = curr['nr_rows'], curr['nr_cols'], curr['nr_nzeros']
            avg_bw = feat['bw-scaled-avg'] * n
            std_bw = feat['bw-scaled-std'] * n
            rows.append({
                'matrix_name': matrix_name,
                'distribution': 'unused', 'placement': 'unused',
                'seed': curr['seed'] if seed_from_data else 0,
                'nr_rows': m, 'nr_cols': n, 'nr_nzeros': nz,
                'density': nz / (m * n) * 100.0,
                'mem_footprint': curr['mem_footprint'],
                'mem_range': find_class((nz * (64 + 32) + 32 * (m + 1)) / (8 * 1024 * 1024)),
                'avg_nnz_per_row': feat['nnz-r-avg'],
                'std_nnz_per_row': feat['nnz-r-std'],
                'avg_bw': avg_bw, 'std_bw': std_bw,
                'avg_bw_scaled': avg_bw / n, 'std_bw_scaled': std_bw / n,
                'skew': feat['skew_coeff'],
                'avg_num_neighbours': feat['num-neigh-avg'],
                'cross_row_similarity': feat['cross_row_sim-avg'],
                'format_name': curr['format_name'],
                'time': curr['time'], 'gflops': curr['gflops'],
                'W_avg': curr['W_avg'], 'J_estimated': curr['J_estimated'],
            })

    result = pd.DataFrame(rows, columns=header_names)
    result['System'] = system
    result['Arch'] = arch

    # V100 mem_footprint adjustment for CSR5_9 format
    if system == 'NVIDIA-V100':
        mask = result['format_name'] == 'CSR5_9'
        result.loc[mask, 'mem_footprint'] = (
            (result.loc[mask, 'mem_footprint'] - 4 * (result.loc[mask, 'nr_rows'] + result.loc[mask, 'nr_cols']))
            / (1024 * 1024.0)
        )

    fname2 = fname.replace('.csv', '_modified_features.csv')
    result.to_csv(f'../benchmark_results/{fname2}', header=False, index=False)
    return result

---
# GPU data

In [4]:
# (arch, system, fname, schema, key_for_merge_or_None)
gpu_configs = [
    ('GPU', 'NVIDIA-P100',    'vulcan-P100/vulcan-P100_dtype-D_run_validation_matrices.csv',   SCHEMA_OLD,  'GPU_P100'),
    ('GPU', 'NVIDIA-V100',    'vulcan-V100/vulcan-V100_dtype-D_run_validation_matrices.csv',   SCHEMA_V100, 'GPU_V100'),
    ('GPU', 'NVIDIA-A100',    'epyc5-A100/epyc5-A100_dtype-D_run_validation_matrices.csv',     SCHEMA_OLD,  'GPU_A100'),
    ('GPU', 'NVIDIA-H100',    'grace1-H100/grace1-H100_validation_matrices_d.csv',             SCHEMA_NEW,  'GPU_H100'),
    ('GPU', 'NVIDIA-H100',    'grace1-H100/grace1-H100_validation_matrices_f.csv',             SCHEMA_NEW,  None),  # FP32
    ('GPU', 'AMD-MI250',      'amd-mi250/amd-mi250_validation_matrices_d.csv',                 SCHEMA_NEW,  'GPU_MI250'),
    ('GPU', 'NVIDIA-RTX3060', 'dungani-rtx3060/dungani-rtx3060_validation_matrices_d.csv',     SCHEMA_NEW,  'GPU_RTX3060'),
    ('GPU', 'NVIDIA-RTX3060', 'dungani-rtx3060/dungani-rtx3060_validation_matrices_f.csv',     SCHEMA_NEW,  None),  # FP32
]

gpu_results = {}
for arch, system, fname, schema, key in gpu_configs:
    df = pd.read_csv(f'../benchmark_results/{fname}', names=schema)
    result = create_complete_csv(df, system, arch, fname, seed_from_data=True)
    if key is not None:
        gpu_results[key] = result
    print(f"  {system:20s} {fname.split('/')[-1]:55s}  rows={len(result)}")

  NVIDIA-P100          vulcan-P100_dtype-D_run_validation_matrices.csv          rows=622
  NVIDIA-V100          vulcan-V100_dtype-D_run_validation_matrices.csv          rows=930
  NVIDIA-A100          epyc5-A100_dtype-D_run_validation_matrices.csv           rows=925
  NVIDIA-H100          grace1-H100_validation_matrices_d.csv                    rows=2050
  NVIDIA-H100          grace1-H100_validation_matrices_f.csv                    rows=260
  AMD-MI250            amd-mi250_validation_matrices_d.csv                      rows=2080
  NVIDIA-RTX3060       dungani-rtx3060_validation_matrices_d.csv                rows=260
  NVIDIA-RTX3060       dungani-rtx3060_validation_matrices_f.csv                rows=260


---
# CPU data

In [5]:
cpu_configs = [
    ('CPU', 'AMD-EPYC-24',       f'amd-epyc1/amd-epyc1_validation_matrices_t{Epyc_threads}_d.csv',               'CPU_Epyc1'),
    ('CPU', 'AMD-EPYC-64',       f'amd-epyc7763/amd-epyc7763_validation_matrices_t{Hawk_threads}_d.csv',         'CPU_Epyc64'),
    ('CPU', 'INTEL-XEON-14',     f'intel-gold2/intel-gold2_validation_matrices_t{Xeon_threads}_d.csv',           'CPU_Gold2'),
    ('CPU', 'INTEL-ICY-16',      f'intel-icy3/intel-icy3_validation_matrices_t{Icy_threads}_d.csv',              'CPU_Icy3'),
    ('CPU', 'INTEL-SAPPHIRE-56', f'intel-sapphire/intel-sapphire_validation_matrices_t{Sapphire_threads}_d.csv', 'CPU_Sapphire'),
    ('CPU', 'ARM-NEON-80',       f'arm/arm_validation_matrices_t{Arm_threads}_d.csv',                            'CPU_Arm'),
    ('CPU', 'ARM-GRACE-72',      f'grace1-arm/grace1-arm_validation_matrices_t{Grace_threads}_d.csv',            'CPU_Arm_Grace'),
    ('CPU', 'ARM-GRACE-72',      f'grace1-arm/grace1-arm_validation_matrices_t{Grace_threads}_f.csv',            None),  # FP32
    ('CPU', 'IBM-POWER9-32',     f'power9-m100/power9-m100_validation_matrices_t{Power9_threads}_d.csv',         'CPU_Power9'),
]

cpu_results = {}
for arch, system, fname, key in cpu_configs:
    df = pd.read_csv(f'../benchmark_results/{fname}', names=SCHEMA_NEW)
    result = create_complete_csv(df, system, arch, fname, seed_from_data=False)
    if key is not None:
        cpu_results[key] = result
    print(f"  {system:20s} {fname.split('/')[-1]:55s}  rows={len(result)}")

  AMD-EPYC-24          amd-epyc1_validation_matrices_t24_d.csv                  rows=1870
  AMD-EPYC-64          amd-epyc7763_validation_matrices_t64_d.csv               rows=2470
  INTEL-XEON-14        intel-gold2_validation_matrices_t14_d.csv                rows=1620
  INTEL-ICY-16         intel-icy3_validation_matrices_t16_d.csv                 rows=412
  INTEL-SAPPHIRE-56    intel-sapphire_validation_matrices_t56_d.csv             rows=2470
  ARM-NEON-80          arm_validation_matrices_t80_d.csv                        rows=1224
  ARM-GRACE-72         grace1-arm_validation_matrices_t72_d.csv                 rows=1550
  ARM-GRACE-72         grace1-arm_validation_matrices_t72_f.csv                 rows=260
  IBM-POWER9-32        power9-m100_validation_matrices_t32_d.csv                rows=601


---
# read FPGA Data (SKIP FOR NOW...)

In [6]:
# def create_complete_fpga_csv(fpga_dataframe, system, arch):
#     inputvaldata_FPGA = pd.DataFrame(columns=header_names)
#     for matrix_name in matrix_names:
#         pin_df = fpga_dataframe[fpga_dataframe['matrix'] == matrix_name]
#         ... [entire function body restored as comments] ...
#     inputvaldata_FPGA['System'] = system
#     inputvaldata_FPGA['Arch'] = arch
#     return inputvaldata_FPGA

In [7]:
# arch, system = 'FPGA', 'Alveo-U280'
# fname = 'alveo-u280/alveo-u280_spmv_validation_matrices_dtype-D.csv'
# fpga_data = pd.read_csv('../benchmark_results/%s' % fname, names = ['matrix','nr_rows', ...])
# fpga_data = create_complete_fpga_csv(fpga_data, system, arch)
# fname2 = fname.replace('.csv', '_modified_features.csv')
# fpga_data.to_csv('../benchmark_results/%s' % fname2, header=False, index=False)


---
# Assign memory/cache bandwidth specifications

In [8]:
bw_specs = {
    # GPU: (mem_bw_gbytes_s, cache_bw_gbytes_s)
    'GPU_P100':      (464,  464),
    'GPU_V100':      (760,  760),
    'GPU_A100':      (1350, 1350),
    'GPU_H100':      (3300, 3300),
    'GPU_RTX3060':   (360,  360),
    'GPU_MI250':     (1313, 1313),
    # CPU:
    'CPU_Epyc1':     (50,   700),
    'CPU_Epyc64':    (120,  900),   # cache_bw needs verification
    'CPU_Gold2':     (55,   300),
    'CPU_Icy3':      (75,   350),
    'CPU_Sapphire':  (250,  1200),
    'CPU_Arm':       (122,  820),
    'CPU_Arm_Grace': (450,  1500),
    'CPU_Power9':    (100,  600),   # approximate values
}

all_results = {**gpu_results, **cpu_results}
for key, df in all_results.items():
    mem_bw, cache_bw = bw_specs[key]
    df['mem_bw_gbytes_s'] = mem_bw
    df['cache_bw_gbytes_s'] = cache_bw

In [9]:
# fname = 'alveo-u280/PADDED-alveo-u280_spmv_validation_matrices_dtype-D_modified_features.csv'
# inputvaldata_FPGA = pd.read_csv('../benchmark_results/%s' % fname, names = header_names, index_col=False)
# inputvaldata_FPGA['mem_bw_gbytes_s'] = 287.5 #  20/32 * 460 = 287.5 GB/s
# inputvaldata_FPGA['cache_bw_gbytes_s'] = 287.5

---
# Merge the results

In [10]:
%%time
inputvaldata = pd.concat(list(all_results.values()))
# inputvaldata = pd.concat([inputvaldata, inputvaldata_FPGA])
print(inputvaldata.shape)

# Group per reps, take mean
groupvalreps = inputvaldata.groupby(
    ['matrix_name','distribution','placement','seed',
     'nr_rows','nr_cols','nr_nzeros','density','mem_footprint','mem_range',
     'avg_nnz_per_row','std_nnz_per_row',
     'avg_bw','std_bw','avg_bw_scaled','std_bw_scaled',
     'skew','avg_num_neighbours','cross_row_similarity',
     'format_name','System', 'Arch', 'mem_bw_gbytes_s','cache_bw_gbytes_s']
).mean().reset_index().reindex(columns=header_names_ext)

group_val_system_best = groupvalreps

(19084, 32)
CPU times: user 33.3 ms, sys: 4.49 ms, total: 37.7 ms
Wall time: 43.7 ms


---
# Group by 'best-of' format_name for each device
# Skip this step if you want to plot per-format validation plots

In [11]:
%%time
# Group per system, take best
# Note: mem_footprint excluded from groupby because CSR5 reports different values for the same matrix
groupval_system = groupvalreps.groupby(
    ['matrix_name','distribution','placement','seed',
     'nr_rows','nr_cols','nr_nzeros','density','mem_range',
     'avg_nnz_per_row','std_nnz_per_row',
     'avg_bw','std_bw','avg_bw_scaled','std_bw_scaled',
     'skew','avg_num_neighbours','cross_row_similarity',
     'System','Arch', 'mem_bw_gbytes_s','cache_bw_gbytes_s'], as_index=False)

reslist = []
for desc, experiment in groupval_system:
    best_format = experiment['format_name'].iloc[experiment['gflops'].argmax()]
    outrow = experiment[experiment['format_name'] == best_format]
    outrow = outrow[header_names_ext]
    reslist.append(outrow.values.tolist()[0])

group_val_system_best = pd.DataFrame(reslist, columns=header_names_ext)
print(group_val_system_best.shape)

(726, 32)
CPU times: user 1.14 s, sys: 12.3 ms, total: 1.15 s
Wall time: 1.17 s


# V100 mem_footprint manual corrections

In [12]:
v100_footprint_fixes = {
    'scircuit': 11.63,
    'mac_econ_fwd500': 15.36,
    'raefsky3': 17.12,
    'rgg_n_2_17_s0': 17.18,
    'bbmat': 20.42,
    'appu': 21.26,
    'conf5_4-8x8-15': 22.13,
    'mc2depi': 26.04,
    'rma10': 27.35,
    'cop20k_A': 30.5,
    'thermomech_dK': 33.35,
    'webbase-1M': 39.35,
    'cant': 46.1,
    'ASIC_680k': 46.91,
    'roadNet-TX': 49.3,
    'pdb1HYS': 49.86,
    'TSOPF_RS_b300_c3': 50.67,
    'Chebyshev4': 61.8,
    'consph': 69.1,
    'com-Youtube': 72.71,
    'rajat30': 73.13,
    'radiation': 88.26,
    'Stanford_Berkeley': 89.39,
    'shipsec1': 89.95,
    'PR02R': 94.29,
    'CurlCurl_2': 105.18,
    'gupta3': 106.76,
    'mip1': 118.73,
    'rail4284': 129.15,
    'pwtk': 133.98,
    'crankseg_2': 162.16,
    'Si41Ge41H72': 172.5,
    'TSOPF_RS_b2383': 185.21,
    'in-2004': 198.88,
    'Ga41As41H72': 212.61,
    'eu-2005': 223.42,
    'wikipedia-20051105': 232.29,
    'kron_g500-logn18': 243.22,
    'human_gene1': 282.41,
    'delaunay_n22': 304,
    'GL7d20': 347.58,
    'sx-stackoverflow': 424.58,
    'dgreen': 442.43,
    'mawi_201512012345': 506.18,
    'ldoor': 536.04,
    'dielFilterV2real': 559.9,
    'circuit5M': 702.4,
    'soc-LiveJournal1': 808.06,
    'bone010': 823.92,
    'audikw_1': 892.25,
    'cage15': 1154.91,
    'kmer_V2a': 1551.42,
}

v100_mask = group_val_system_best['System'] == 'NVIDIA-V100'
for matrix_name, footprint in v100_footprint_fixes.items():
    mask = v100_mask & (group_val_system_best['matrix_name'] == matrix_name)
    group_val_system_best.loc[mask, 'mem_footprint'] = footprint

---
# Export final CSVs

In [13]:
groupvalreps.to_csv('validation_real_benchmarks_all-devices_all.csv', sep=',', header=True, index=False)
group_val_system_best.to_csv('validation_real_all-devices_best-of.csv', sep=',', header=True, index=False)
print("Done. Exported validation_real_benchmarks_all-devices_all.csv and validation_real_all-devices_best-of.csv")

Done. Exported validation_real_benchmarks_all-devices_all.csv and validation_real_all-devices_best-of.csv
